<a href="https://colab.research.google.com/github/q3eq3e/Artificial-neural-networks-lab/blob/main/cw2/cw2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [64]:
import zipfile
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

In [65]:
label_sale_price = lambda amount: 0 if amount <= 100000 else 2 if amount > 350000 else 1

train_data, test_data = (
    pd.read_csv('train_data.csv'),
    pd.read_csv('test_data.csv'),
)

In [66]:
# create class and delete SalePrice

train_data['Target'] = train_data['SalePrice'].apply(label_sale_price)

train_data = train_data.drop('SalePrice', axis=1)

train_data

,YearBuilt,Size(sqf),Floor,HallwayType,HeatingType,AptManageType,N_Parkinglot(Ground),N_Parkinglot(Basement),TimeToBusStop,TimeToSubway,N_manager,N_elevators,SubwayStation,N_FacilitiesInApt,N_FacilitiesNearBy(Total),N_SchoolNearBy(Total),Target
0,2006,814,3,terraced,individual_heating,management_in_trust,111.0,184.0,5min~10min,10min~15min,3.0,0.0,Kyungbuk_uni_hospital,5,6.0,9.0,1
1,1985,587,8,corridor,individual_heating,self_management,80.0,76.0,0~5min,5min~10min,2.0,2.0,Daegu,3,12.0,4.0,0
2,1985,587,6,corridor,individual_heating,self_management,80.0,76.0,0~5min,5min~10min,2.0,2.0,Daegu,3,12.0,4.0,0
3,2006,2056,8,terraced,individual_heating,management_in_trust,249.0,536.0,0~5min,0-5min,5.0,11.0,Sin-nam,5,3.0,7.0,2
4,1992,644,2,mixed,individual_heating,self_management,142.0,79.0,5min~10min,15min~20min,4.0,8.0,Myung-duk,3,9.0,14.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4119,2007,1928,24,terraced,individual_heating,management_in_trust,0.0,1270.0,0~5min,0-5min,14.0,16.0,Kyungbuk_uni_hospital,10,9.0,10.0,2
4120,2015,644,22,terraced,individual_heating,management_in_trust,102.0,400.0,0~5min,5min~10min,5.0,10.0,Daegu,7,7.0,11.0,1
4121,2007,868,20,terraced,individual_heating,management_in_trust,0.0,1270.0,0~5min,0-5min,14.0,16.0,Kyungbuk_uni_hospital,10,9.0,10.0,2
4122,1978,1327,1,corridor,individual_heating,self_management,87.0,0.0,0~5min,0-5min,1.0,4.0,Kyungbuk_uni_hospital,3,7.0,11.0,1


In [67]:
# target variable
y_train = train_data['Target']
X_train = train_data.drop('Target', axis=1)
X_test = test_data.copy()

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train,
    y_train.values,
    test_size=0.2,
    random_state=42,
    stratify=y_train.values,
)

numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# pipeline for numeric values: fills with median then scales
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# pipeline for categorical: fills with none then onehot encodes
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# merge two pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

X_train_scaled = preprocessor.fit_transform(X_train_split)
X_val_scaled = preprocessor.transform(X_val_split)
X_test_scaled = preprocessor.transform(X_test)

print(f"Wymiary X_train_scaled: {X_train_scaled.shape}")
print(f"Wymiary X_train_scaled: {X_val_scaled.shape}")
print(f"Wymiary X_test_scaled: {X_test_scaled.shape}")

Wymiary X_train_scaled: (3299, 33)
Wymiary X_train_scaled: (825, 33)
Wymiary X_test_scaled: (1767, 33)


In [68]:
class HousePriceNet(nn.Module):
    def __init__(self, input_size):
        super(HousePriceNet, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(0.5),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Dropout(0.5),

            nn.Linear(64, 3)    # 3 because (0, 1, 2) = (cheap, medium, expensive)
        )

    def forward(self, x):
        return self.net(x)

In [69]:
X_train_tensor, y_train_tensor = (
    torch.FloatTensor(X_train_scaled),  # zmienic na scaled
    torch.LongTensor(y_train_split),
)

X_val_tensor, y_val_tensor = (
    torch.FloatTensor(X_val_scaled),
    torch.LongTensor(y_val_split),
)

dataset = TensorDataset(X_train_tensor, y_train_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
weights_tensor = torch.FloatTensor(class_weights)

weights_tensor

tensor([2.4460, 0.4594, 2.4117])

In [73]:
num_cols = X_train_scaled.shape[1]
model = HousePriceNet(input_size=num_cols)

# loss function
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

REGULARIZATION_LAMBDA = 1e-3

# adam optimizer
learning_rate = 0.0005
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# learn loop
epochs = 200
model.train()

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_X)

        # L2 regularization
        sum = 0
        for name, val in list(model.named_parameters()):
            if ".weight" in name:
                sum += torch.sum(torch.square(val))

        regularization = sum * REGULARIZATION_LAMBDA
        loss = criterion(outputs, batch_y) + regularization
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    # validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_tensor)

        # L2 regularization
        sum = 0
        for name, val in list(model.named_parameters()):
            if ".weight" in name:
                sum += torch.sum(torch.square(val))

        regularization = sum * REGULARIZATION_LAMBDA
        val_loss = criterion(val_outputs, y_val_tensor) + regularization
        _, val_predictions = torch.max(val_outputs, 1)

        # balanced accuracy for validation
        val_bal_acc = balanced_accuracy_score(y_val_tensor.numpy(), val_predictions.numpy())

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}] | "
              f"Train Loss: {epoch_loss/len(dataloader):.4f} | "
              f"Val Loss: {val_loss.item():.4f} | "
              f"Val Balanced Acc: {val_bal_acc*100:.2f}%")

print("Training complete")

Epoch [10/200] | Train Loss: 0.6128 | Val Loss: 0.5506 | Val Balanced Acc: 87.81%
Epoch [20/200] | Train Loss: 0.5306 | Val Loss: 0.4852 | Val Balanced Acc: 88.72%
Epoch [30/200] | Train Loss: 0.4950 | Val Loss: 0.4491 | Val Balanced Acc: 88.27%
Epoch [40/200] | Train Loss: 0.4698 | Val Loss: 0.4258 | Val Balanced Acc: 89.05%
Epoch [50/200] | Train Loss: 0.4470 | Val Loss: 0.4007 | Val Balanced Acc: 89.51%
Epoch [60/200] | Train Loss: 0.4311 | Val Loss: 0.3926 | Val Balanced Acc: 89.34%
Epoch [70/200] | Train Loss: 0.4218 | Val Loss: 0.3860 | Val Balanced Acc: 89.23%
Epoch [80/200] | Train Loss: 0.4304 | Val Loss: 0.3823 | Val Balanced Acc: 88.48%
Epoch [90/200] | Train Loss: 0.4065 | Val Loss: 0.3728 | Val Balanced Acc: 89.25%
Epoch [100/200] | Train Loss: 0.4071 | Val Loss: 0.3770 | Val Balanced Acc: 88.91%
Epoch [110/200] | Train Loss: 0.4064 | Val Loss: 0.3646 | Val Balanced Acc: 89.19%
Epoch [120/200] | Train Loss: 0.3998 | Val Loss: 0.3651 | Val Balanced Acc: 89.18%
Epoch [130/20

In [74]:
model.eval()

X_test_tensor = torch.FloatTensor(X_test_scaled)

with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predictions = torch.max(test_outputs, 1)

final_preds = predictions.numpy()

# save predictions to pred.csv (no header, one integer column)
pd.DataFrame(final_preds).to_csv('pred.csv', index=False, header=False)

print(f"Saved {len(final_preds)} predictions to pred.csv")
print(f"Class distribution: {pd.Series(final_preds).value_counts().sort_index().to_dict()}")


Saved 1767 predictions to pred.csv
Class distribution: {0: 402, 1: 1020, 2: 345}


In [75]:
model.eval()

with torch.no_grad():
    val_outputs = model(X_val_tensor)
    _, val_predictions = torch.max(val_outputs, 1)

val_preds_np = val_predictions.numpy()
y_val_np = y_val_tensor.numpy()

print("\n--- Report on validation set ---")
print("Balanced Accuracy:")
print(f"{balanced_accuracy_score(y_val_np, val_preds_np) * 100:.2f}%\n")

print("Detailed classification report:")
print(classification_report(y_val_np, val_preds_np))


--- Report on validation set ---
Balanced Accuracy:
89.41%

Detailed classification report:
              precision    recall  f1-score   support

           0       0.58      0.98      0.73       112
           1       0.98      0.75      0.85       599
           2       0.61      0.95      0.74       114

    accuracy                           0.81       825
   macro avg       0.73      0.89      0.78       825
weighted avg       0.88      0.81      0.82       825



In [76]:
# pack into zip file

DAY = "sroda"
AUTHOR_1 = "BagińskiJakub"
AUTHOR_2 = "GórskiKacper"
CODE_FILE = "cw2.ipynb"

autorzy = sorted([AUTHOR_1, AUTHOR_2])
nazwa_zip = f"{DAY}_{autorzy[0]}_{autorzy[1]}.zip"

with zipfile.ZipFile(nazwa_zip, 'w') as zipf:
    if os.path.exists('pred.csv'):
        zipf.write('pred.csv')
    else:
        print("ERROR: pred.csv not found!")

    if os.path.exists(CODE_FILE):
        zipf.write(CODE_FILE)
        print(f"Package ready: {nazwa_zip}")
    else:
        print(f"ERROR: {CODE_FILE} not found!")



Package ready: sroda_BagińskiJakub_GórskiKacper.zip
